In [ ]:
from google.colab import files
uploaded = files.upload()

Saving reportes_agua_2024_01.csv to reportes_agua_2024_01.csv


In [ ]:
import pandas as pd

# Cargar el CSV
df = pd.read_csv('reportes_agua_2024_01.csv')

# Ver las columnas disponibles
print(df.columns.tolist())
#Imprimimos 3 para simplemente observar el dataframe
print(df.head(3))

['folio_incidente', 'fecha_registro_incidente', 'id_reporte', 'fecha_reporte', 'hora_reporte', 'clasificacion', 'reporte', 'medio_recepcion', 'alcaldia_catalogo', 'colonia_catalogo', 'longitud', 'latitud']
   folio_incidente fecha_registro_incidente       id_reporte fecha_reporte  \
0  I-20220101-0001               2022-01-02  R-20220101-0105    2022-01-01   
1  I-20220101-0002               2022-01-02  R-20220101-0106    2022-01-01   
2  I-20220101-0003               2022-01-02  R-20220101-0103    2022-01-01   

  hora_reporte clasificacion            reporte          medio_recepcion  \
0     18:33:08  Agua Potable               Fuga  Ciudadano (Call Center)   
1     18:36:38       Drenaje  Drenaje Obstruido  Ciudadano (Call Center)   
2     18:12:39  Agua Potable               Fuga  Ciudadano (Call Center)   

  alcaldia_catalogo                                   colonia_catalogo  \
0          Coyoacán                         Cuadrante De San Francisco   
1        Xochimilco         

In [ ]:
#Imprimimos todas las categorías que hay de reportes
print(df['reporte'].unique())

['Fuga' 'Drenaje Obstruido' 'Falta de agua' 'Falta de tapa en válvula'
 'Brote en aguas negras' 'Mala calidad' 'Mal uso' 'Hundimiento'
 'Coladera sin tapa' 'Rejilla de piso' 'Pozo de visita' 'Reconstrucción'
 'Encharcamiento' 'Rehabilitación' 'Socavon' 'Boca de tormenta' 'Otros'
 'Hidrocarburos' 'Solicitud' 'Lavado y Desinfección']


In [ ]:
#Ponemos encharcamineto en minúsculas y verificamos si contiene o no la clave encharcamiento
df_ench = df[df['reporte'].str.lower().str.contains('encharcamiento', na=False)].copy()

In [ ]:
# Eliminar filas sin coordenadas
df_ench = df_ench.dropna(subset=['latitud', 'longitud'])

# Convertir a numérico (si hay errores, se convierten en NaN y los borramos)
df_ench['latitud'] = pd.to_numeric(df_ench['latitud'], errors='coerce')
df_ench['longitud'] = pd.to_numeric(df_ench['longitud'], errors='coerce')
#Eliminamos filas con coordenas de latitud y longitud con valores de tipo NaN
df_ench = df_ench.dropna(subset=['latitud', 'longitud'])

# Redondear a 3 decimales (precisión ~111 metros)
df_ench['lat_round'] = df_ench['latitud'].round(3)
df_ench['lon_round'] = df_ench['longitud'].round(3)

print(f'Coordenadas válidas después de limpiar: {len(df_ench)}')
print(df_ench)

Coordenadas válidas después de limpiar: 3109
        folio_incidente fecha_registro_incidente       id_reporte  \
2487    I-20220106-0533               2022-01-07  R-20220106-0449   
5802    I-20220116-0175               2022-01-16  R-20220116-0188   
28873   I-20220316-0432               2022-03-16  R-20220316-0473   
28983   I-20220317-0011               2022-03-17  R-20220316-0469   
40899   I-20220408-0433               2022-04-08  R-20220408-0461   
...                 ...                      ...              ...   
280407  I-20231122-0524               2023-11-23  R-20231122-0657   
280410  I-20231122-0527               2023-11-23  R-20231122-0653   
281392  I-20231124-0359               2023-11-24  R-20231124-0412   
286500  I-20231205-0418               2023-12-06  R-20231205-0522   
286509  I-20231205-0425               2023-12-06  R-20231205-0531   

       fecha_reporte hora_reporte clasificacion         reporte  \
2487      2022-01-06     18:18:20       Drenaje  Encharcami

In [ ]:
#Agrupamos los pares de latitud y longitud redondeados en grupos
puntos_criticos = df_ench.groupby(['lat_round', 'lon_round']).agg(
    frecuencia=('lat_round', 'count') #Contamos cuantos encharcamientos hay para esas mismas coordenadas y agregamos una columna nueva de frecuencia
).reset_index().sort_values('frecuencia', ascending=False) #
print(puntos_criticos)
# Ver los 10 primeros
print(puntos_criticos.head(10))

     lat_round  lon_round  frecuencia
835     19.355    -99.084          23
186     19.284    -99.220          18
575     19.325    -99.211          17
202     19.285    -99.164          15
577     19.325    -99.196          14
..         ...        ...         ...
13      19.193    -99.014           1
12      19.193    -99.015           1
11      19.192    -99.090           1
10      19.191    -99.147           1
9       19.190    -99.003           1

[1994 rows x 3 columns]
      lat_round  lon_round  frecuencia
835      19.355    -99.084          23
186      19.284    -99.220          18
575      19.325    -99.211          17
202      19.285    -99.164          15
577      19.325    -99.196          14
145      19.276    -98.996          13
1960     19.538    -99.141          12
273      19.291    -99.174          12
1984     19.549    -99.145          10
1688     19.463    -99.155          10


In [ ]:
# Obtener la colonia más reportada en cada punto
colonia_por_punto = df_ench.groupby(['lat_round', 'lon_round'])['colonia_catalogo'].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else 'Desconocida'
).reset_index()
print(colonia_por_punto)
# Unimos las 10 primeras filas (que son las de mayor frecuencia) de la tabla de colonias con la tabla de frecuencias
top10 = puntos_criticos.head(10).merge(colonia_por_punto, on=['lat_round', 'lon_round'], how='left')
print(top10[['lat_round', 'lon_round', 'frecuencia', 'colonia_catalogo']])

      lat_round  lon_round         colonia_catalogo
0        -9.190    -75.015              Desconocida
1        19.130    -99.180  Pueblo Parres El Guarda
2        19.131    -99.182              Desconocida
3        19.131    -99.180  Pueblo Parres El Guarda
4        19.132    -99.182  Pueblo Parres El Guarda
...         ...        ...                      ...
1989     19.552    -99.143            Loma La Palma
1990     19.552    -99.142            Loma La Palma
1991     19.554    -99.136    Cuautepec Barrio Alto
1992     19.557    -99.133    Cuautepec Barrio Alto
1993     21.281    -89.665              Desconocida

[1994 rows x 3 columns]
   lat_round  lon_round  frecuencia  \
0     19.355    -99.084          23   
1     19.284    -99.220          18   
2     19.325    -99.211          17   
3     19.285    -99.164          15   
4     19.325    -99.196          14   
5     19.276    -98.996          13   
6     19.538    -99.141          12   
7     19.291    -99.174          12   


In [ ]:
print(top10[['lat_round', 'lon_round', 'frecuencia', 'colonia_catalogo']])

   lat_round  lon_round  frecuencia  \
0     19.355    -99.084          23   
1     19.284    -99.220          18   
2     19.325    -99.211          17   
3     19.285    -99.164          15   
4     19.325    -99.196          14   
5     19.276    -98.996          13   
6     19.538    -99.141          12   
7     19.291    -99.174          12   
8     19.549    -99.145          10   
9     19.463    -99.155          10   

                                colonia_catalogo  
0                              Barrio San Miguel  
1                             Colinas Del Ajusco  
2                          Jardines Del Pedregal  
3                                        La Joya  
4                          Jardines Del Pedregal  
5  San Jose Tlahuac Del Pueblo San Pedro Tlahuac  
6                           Zona Escolar Oriente  
7                                        Tlalpan  
8                                  Loma La Palma  
9                                        Atlampa  


In [ ]:
rutas_waypoints = {
    'R1': [  # Viaducto + Av. Revolución
        (-99.180, 19.400),
        (-99.178, 19.398),
        (-99.175, 19.395),
        (-99.172, 19.402),
        (-99.170, 19.410)
    ],
    'R2': [  # Circuito Interior
        (-99.150, 19.450),
        (-99.145, 19.440),
        (-99.140, 19.430),
        (-99.137, 19.425),
        (-99.135, 19.420)
    ],
    'R3': [  # Periférico Sur + Eje 10
        (-99.200, 19.360),
        (-99.195, 19.355),
        (-99.190, 19.350),
        (-99.185, 19.345),
        (-99.180, 19.340)
    ],
    'R4': [  # Eje Central + Reforma
        (-99.140, 19.430),
        (-99.137, 19.428),
        (-99.135, 19.425),
        (-99.132, 19.420),
        (-99.130, 19.415)
    ]
}

In [ ]:
import numpy as np
#Definimos una función que dados dos puntos P1 = (lon1, lat1) y P2 = (lon2, lat2)
#Calcula la distancia entre ellos considerando que la geometría del espacio sobre el cual están los puntos no es un plano recto
def haversine(lon1, lat1, lon2, lat2):
    R = 6371000  # radio de la Tierra en metros
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

In [ ]:
#Definimos una función que dado un punto y una ruta (conformada por varios puntos y los segmentos que los unen)
#Calcula la distancia mínima a dicha ruta (formada por polilíneas)
def distancia_punto_a_ruta(punto, ruta):
    min_dist = float('inf')
    for i in range(len(ruta)-1):
        a = np.array(ruta[i])
        b = np.array(ruta[i+1])
        p = np.array(punto)
        ab = b - a
        ap = p - a
        t = np.dot(ap, ab) / np.dot(ab, ab)
        t = max(0, min(1, t))
        closest = a + t * ab
        dist = haversine(p[0], p[1], closest[0], closest[1])
        min_dist = min(min_dist, dist)
    return min_dist

In [ ]:
# Asignamos cada punto crítico a la ruta MÁS CERCANA (en vez de un buffer fijo)
# Esto evita el problema de elegir un umbral arbitrario y refleja mejor
# el riesgo relativo de cada corredor vial.

puntos_criticos = [(lon, lat) for lat, lon in top10[['lat_round','lon_round']].values]

conteos_por_ruta = {ruta: 0 for ruta in rutas_waypoints}
#print(conteos_por_ruta)
distancia_min_por_punto = []  # opcional: para documentar

for (lon, lat) in puntos_criticos:
    dists = {nombre: distancia_punto_a_ruta((lon, lat), wp) for nombre, wp in rutas_waypoints.items()}
    ruta_mas_cercana = min(dists, key=dists.get) #Comparamos con key= elementos y no claves, pero devolvemos la clave con la distancia mínima
    conteos_por_ruta[ruta_mas_cercana] += 1
    distancia_min_por_punto.append((lat, lon, ruta_mas_cercana, round(dists[ruta_mas_cercana], 1)))

print("Conteos por ruta:", conteos_por_ruta)
#print("\nDetalle (lat, lon, ruta asignada, distancia en m):")
#for d in distancia_min_por_punto:
#    print(d)

#Lo siguiente es solo para vsualizar mejor los datos
distancia_min_por_punto_df = pd.DataFrame(distancia_min_por_punto, columns=['lat', 'lon', 'ruta', 'distancia_m'])
print(distancia_min_por_punto_df)

Conteos por ruta: {'R1': 0, 'R2': 3, 'R3': 6, 'R4': 1}
      lat     lon ruta  distancia_m
0  19.355 -99.084   R4       8233.6
1  19.284 -99.220   R3       7509.6
2  19.325 -99.211   R3       3516.3
3  19.285 -99.164   R3       6342.0
4  19.325 -99.196   R3       2369.7
5  19.276 -98.996   R3      20578.8
6  19.538 -99.141   R2       9830.5
7  19.291 -99.174   R3       5484.8
8  19.549 -99.145   R2      11020.8
9  19.463 -99.155   R2       1537.7


In [ ]:
# ========== GENERAR MATRIZ 4x4 Y CSV ==========
"""
max_conteo = max(conteos_por_ruta.values())  # la ruta con más puntos críticos cerca
print(max_conteo)
usuarios = ['U1', 'U2', 'U3', 'U4']
rutas = ['R1', 'R2', 'R3', 'R4']
nombres_usuarios = ['Polanco→Condesa', 'Coyoacán→Centro', 'Iztapalapa→Santa Fe', 'Lindavista→Roma']
nombres_rutas = ['Viaducto+Revolución', 'Circuito Interior', 'Periférico Sur', 'Eje Central']

filas = []
for u, nombre_u in zip(usuarios, nombres_usuarios):
    for r, nombre_r in zip(rutas, nombres_rutas):
        conteo = conteos_por_ruta[r]
        riesgo = conteo / max_conteo          # 0 = sin riesgo, 1 = riesgo máximo
        score = round(1 - riesgo, 2)          # score = compatibilidad (1 = mejor, 0 = peor)
        filas.append([u, r, score, nombre_u, nombre_r]) #Usuario, Ruta, Score, nombre del usuario, nombre de la ruta

df_final = pd.DataFrame(filas, columns=['a_id', 'b_id', 'score', 'a_nombre', 'b_nombre'])
print(df_final)

df_final.to_csv('dataset_real_4x4.csv', index=False)
from google.colab import files
#files.download('dataset_real_4x4.csv')
"""

"\nmax_conteo = max(conteos_por_ruta.values())  # la ruta con más puntos críticos cerca\nprint(max_conteo)\nusuarios = ['U1', 'U2', 'U3', 'U4']\nrutas = ['R1', 'R2', 'R3', 'R4']\nnombres_usuarios = ['Polanco→Condesa', 'Coyoacán→Centro', 'Iztapalapa→Santa Fe', 'Lindavista→Roma']\nnombres_rutas = ['Viaducto+Revolución', 'Circuito Interior', 'Periférico Sur', 'Eje Central']\n\nfilas = []\nfor u, nombre_u in zip(usuarios, nombres_usuarios):\n    for r, nombre_r in zip(rutas, nombres_rutas):\n        conteo = conteos_por_ruta[r]\n        riesgo = conteo / max_conteo          # 0 = sin riesgo, 1 = riesgo máximo\n        score = round(1 - riesgo, 2)          # score = compatibilidad (1 = mejor, 0 = peor)\n        filas.append([u, r, score, nombre_u, nombre_r]) #Usuario, Ruta, Score, nombre del usuario, nombre de la ruta\n\ndf_final = pd.DataFrame(filas, columns=['a_id', 'b_id', 'score', 'a_nombre', 'b_nombre'])\nprint(df_final)\n\ndf_final.to_csv('dataset_real_4x4.csv', index=False)\nfrom goo

In [ ]:
# ========== GENERAR MATRIZ 4x4 CON SCORES BASADOS EN DISTANCIA ==========

# 1. Seleccionar los 4 puntos críticos más frecuentes como usuarios
top4 = top10.head(4).copy()
top4['punto'] = list(zip(top4['lon_round'], top4['lat_round']))

usuarios = ['U1', 'U2', 'U3', 'U4']
nombres_usuarios = top4['colonia_catalogo'].values  # Usamos la colonia real

# 2. Rutas y sus waypoints (ya las tienes)
rutas = ['R1', 'R2', 'R3', 'R4']
nombres_rutas = ['Viaducto+Revolución', 'Circuito Interior', 'Periférico Sur', 'Eje Central']

# 3. Calcular matriz de distancias (4 usuarios × 4 rutas)
matriz_distancias = np.zeros((4, 4))

for i, (_, row) in enumerate(top4.iterrows()):
    punto = (row['lon_round'], row['lat_round'])
    for j, r in enumerate(rutas):
        dist = distancia_punto_a_ruta(punto, rutas_waypoints[r])
        matriz_distancias[i, j] = dist

print("Matriz de distancias (metros):")
print(pd.DataFrame(matriz_distancias, index=usuarios, columns=rutas).round(1))

# 4. Convertir distancias a scores (compatibilidad)
#    Usamos una función: score = exp(-dist / escala) o una normalización inversa.
#    Aquí usamos: score = 1 - (dist - min) / (max - min)  (normalización inversa)

dist_min = matriz_distancias.min()
dist_max = matriz_distancias.max()
# Evitar división por cero si todas las distancias son iguales (no debería pasar)
rango = dist_max - dist_min
if rango == 0:
    rango = 1.0

matriz_scores = 1 - (matriz_distancias - dist_min) / rango
# Redondear a 2 decimales
matriz_scores = np.round(matriz_scores, 2)

print("\nMatriz de scores (compatibilidad):")
print(pd.DataFrame(matriz_scores, index=usuarios, columns=rutas))

# 5. Generar CSV en formato largo
filas = []
for i, u in enumerate(usuarios):
    for j, r in enumerate(rutas):
        filas.append([
            u,
            r,
            matriz_scores[i, j],
            nombres_usuarios[i],
            nombres_rutas[j]
        ])

df_final = pd.DataFrame(filas, columns=['a_id', 'b_id', 'score', 'a_nombre', 'b_nombre'])
print("\nCSV final:")
print(df_final)

# Guardar y descargar (opcional)
df_final.to_csv('dataset_real_4x4.csv', index=False)
files.download('dataset_real_4x4.csv')

Matriz de distancias (metros):
         R1       R2       R3       R4
U1  10531.1   8991.9  10209.0   8233.6
U2  13214.9  17556.0   7509.6  17359.2
U3   8651.5  13234.3   3516.3  13128.2
U4  12285.8  15316.5   6342.0  14889.0

Matriz de scores (compatibilidad):
      R1    R2    R3    R4
U1  0.50  0.61  0.52  0.66
U2  0.31  0.00  0.72  0.01
U3  0.63  0.31  1.00  0.32
U4  0.38  0.16  0.80  0.19

CSV final:
   a_id b_id  score               a_nombre             b_nombre
0    U1   R1   0.50      Barrio San Miguel  Viaducto+Revolución
1    U1   R2   0.61      Barrio San Miguel    Circuito Interior
2    U1   R3   0.52      Barrio San Miguel       Periférico Sur
3    U1   R4   0.66      Barrio San Miguel          Eje Central
4    U2   R1   0.31     Colinas Del Ajusco  Viaducto+Revolución
5    U2   R2   0.00     Colinas Del Ajusco    Circuito Interior
6    U2   R3   0.72     Colinas Del Ajusco       Periférico Sur
7    U2   R4   0.01     Colinas Del Ajusco          Eje Central
8    U3   R1   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
S_loaded_df = df_final.pivot(index="a_id", columns="b_id", values="score")
print(S_loaded_df)

b_id    R1    R2    R3    R4
a_id                        
U1    0.50  0.61  0.52  0.66
U2    0.31  0.00  0.72  0.01
U3    0.63  0.31  1.00  0.32
U4    0.38  0.16  0.80  0.19
